# 🎯 Daily Grand Data Manager

Welcome! This tool allows you to easily update the `daily_grand_results.csv` dataset with the latest winning numbers from the Daily Grand lottery.

---

## 📝 Instructions

### ✅ Step 1: Copy Raw Text from Website

1. Go to [LotteryExtreme – Daily Grand Winning Numbers](https://www.lotteryextreme.com/canada/dailygrand-winningnumbers)
2. Copy the full block of raw text, including dates and numbers. It should look like this:

    Daily Grand   Jun 19 2025 (2025-06-19 Thu)   Winners  
    25  
    30  
    35  
    36  
    38  
    2  

3. Paste it into the **Raw Input** box (Step 1 of this notebook).

---

### ✅ Step 2: Format It

- Click the **✨ Format** button.
- This will convert the raw input into the following clean format:

    Daily Grand Jun 19 2025  
    25 30 35 36 38 2  
    Daily Grand Jun 16 2025  
    7 11 17 20 21 1

---

### ✅ Step 3: Review & Submit

- Review or edit the formatted text in the **Input** box.
- Click **📥 Add to Dataset**.
- The system will:
  - ✅ Parse and validate each draw
  - ✅ Add new draws to the CSV file
  - ✅ Skip any duplicates
  - ✅ Sort all entries by date (newest first)
  - ✅ Show how many rows were added or skipped

---

## 📦 Output Summary

Once done, the output will confirm:
- 🟢 How many rows were **added**
- ⚪ How many were **duplicates**
- 📊 The **total row count** in the dataset
- 📋 A preview of new draws (if any)

---

> 💡 Tip: Run this notebook every **Monday and Thursday** after a new draw is published.

# 🧠 Formatter & CSV Updater

In [ ]:
import pandas as pd
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

csv_path = "daily_grand_results.csv"

# === Formatter UI ===
formatter_input = widgets.Textarea(
    value='',
    placeholder='Paste raw text from LotteryExtreme here...',
    description='Raw:',
    layout=widgets.Layout(width='100%', height='200px')
)
format_btn = widgets.Button(description="✨ Format", button_style='info')
clear_format_btn = widgets.Button(description="🧹 Clear", button_style='warning')
format_output = widgets.Output()

# === Submission UI ===
input_box = widgets.Textarea(
    value='',
    placeholder='Formatted results will appear here...',
    description='Input:',
    layout=widgets.Layout(width='100%', height='200px')
)
preview_btn = widgets.Button(description="🔍 Preview", button_style='primary')
submit_btn = widgets.Button(description="📥 Add to Dataset", button_style='success')
output_box = widgets.Output()


# === Formatter Logic ===
def clean_lottery_text(raw_text):
    lines = [line.strip() for line in raw_text.split("\n") if line.strip()]
    formatted = []
    i = 0
    while i < len(lines):
        if lines[i].startswith("Daily Grand"):
            date_str = lines[i].split("(")[0].replace("Daily Grand", "").strip()
            formatted.append(f"Daily Grand {date_str}")
            try:
                numbers = list(map(int, lines[i+1:i+7]))
                if len(numbers) == 6:
                    formatted.append(" ".join(map(str, numbers)))
                i += 7
            except:
                i += 1
        else:
            i += 1
    return "\n".join(formatted)


# === Input Parser Logic ===
def parse_input(text):
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    records = []

    for i in range(0, len(lines), 2):
        try:
            date_line = lines[i]
            number_line = lines[i+1]
            date_part = date_line.replace("Daily Grand", "").strip()
            draw_date = datetime.strptime(date_part, "%b %d %Y")
            nums = list(map(int, number_line.strip().split()))
            if len(nums) != 6:
                raise ValueError("Invalid number count")

            record = {
                "day": draw_date.day,
                "month": draw_date.month,
                "year": draw_date.year,
                "dn_1": nums[0],
                "dn_2": nums[1],
                "dn_3": nums[2],
                "dn_4": nums[3],
                "dn_5": nums[4],
                "grand_number": nums[5]
            }
            records.append(record)
        except Exception as e:
            print(f"⚠️ Skipped block due to error: {e}")
            continue

    return pd.DataFrame(records)


# === Button Handlers ===
def on_format(b):
    with format_output:
        clear_output()
        result = clean_lottery_text(formatter_input.value)
        if result.strip():
            input_box.value = result
            print("✅ Format successful. Here's the formatted result:\n")
            print(result)
        else:
            print("❌ No valid draws found.")

def on_clear_format(b):
    with format_output:
        clear_output()
        formatter_input.value = ""

def on_preview(b):
    with output_box:
        clear_output()
        preview_data = parse_input(input_box.value)
        if preview_data.empty:
            print("❌ No valid rows to preview.")
        else:
            print("🔍 Preview of parsed draws:")
            display(preview_data)

def on_submit(b):
    with output_box:
        clear_output()
        print("📥 Submitting formatted results...\n")
        new_data = parse_input(input_box.value)
        if new_data.empty:
            print("❌ No valid rows parsed. Please check formatting.")
            return

        try:
            existing = pd.read_csv(csv_path)
        except FileNotFoundError:
            existing = pd.DataFrame()

        combined = pd.concat([new_data, existing], ignore_index=True)
        combined.drop_duplicates(
            subset=['day', 'month', 'year', 'dn_1', 'dn_2', 'dn_3', 'dn_4', 'dn_5', 'grand_number'],
            inplace=True
        )
        combined.sort_values(by=['year', 'month', 'day'], ascending=False, inplace=True)
        column_order = ['day', 'month', 'year', 'dn_1', 'dn_2', 'dn_3', 'dn_4', 'dn_5', 'grand_number']
        combined = combined[column_order]
        combined.to_csv(csv_path, index=False)

        # Identify truly new rows added
        combined_key = combined[column_order]
        existing_key = existing[column_order] if not existing.empty else pd.DataFrame(columns=column_order)
        merged_diff = pd.merge(combined_key, existing_key, how='outer', indicator=True)
        new_rows = merged_diff[merged_diff['_merge'] == 'left_only'].drop(columns=['_merge'])

        print(f"✅ Submission complete.")
        print(f"🟢 {len(new_rows)} new row(s) added.")
        print(f"⚪ {new_data.shape[0] - len(new_rows)} duplicate(s) ignored.")
        print(f"📊 Dataset now contains {combined.shape[0]} row(s).")

        if not new_rows.empty:
            print("\n📋 New rows added:")
            display(new_rows)


# === Connect Buttons ===
format_btn.on_click(on_format)
clear_format_btn.on_click(on_clear_format)
submit_btn.on_click(on_submit)
preview_btn.on_click(on_preview)


# === Display Interface ===
display(widgets.HTML("<h3>✨ Step 1: Format raw LotteryExtreme input</h3>"))
display(widgets.HBox([formatter_input]), widgets.HBox([format_btn, clear_format_btn]), format_output)

display(widgets.HTML("<h3>📄 Step 2: Review or edit formatted result</h3>"))
display(input_box, preview_btn)

display(widgets.HTML("<h3>📥 Step 3: Submit to dataset</h3>"))
display(submit_btn, output_box)

HTML(value='<h3>✨ Step 1: Format raw LotteryExtreme input</h3>')

Output()

HTML(value='<h3>📄 Step 2: Review or edit formatted result</h3>')

Textarea(value='', description='Input:', layout=Layout(height='200px', width='100%'), placeholder='Formatted r…

Button(button_style='primary', description='🔍 Preview', style=ButtonStyle())

HTML(value='<h3>📥 Step 3: Submit to dataset</h3>')

Button(button_style='success', description='📥 Add to Dataset', style=ButtonStyle())

Output()